# E02 Titanic：缺失机制、特征工程与训练/验证隔离

> Issue #9 | [Kaggle Titanic](https://www.kaggle.com/competitions/titanic) | 本案例用 `seaborn.load_dataset('titanic')` **离线可跑**

## 你将学到
1. **生存率 EDA**：性别、舱位、年龄、票价、登船港口与 `survived` 的关系
2. **缺失机制**：MCAR/MAR/MNAR 直觉 —— Age/Embarked/Deck 不能盲目用全表统计填充
3. **泄漏列**：`alive` 几乎等于标签，必须删除后再建模
4. **特征工程**：Title（有 `name` 则解析，否则用 `who`）、FamilySize、IsAlone、Deck —— **映射只在训练集拟合**
5. **`LeakageDetector`**：编码后扫描异常高相关特征
6. **错误 vs 正确填充**：全表 median 填充会虚高离线分

## 提交
竞赛列名常为 `Survived`；`prediction_type='label'`。


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 1. 加载数据与列概览


In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

raw = sns.load_dataset('titanic').copy()
print('shape', raw.shape)
print('columns:', list(raw.columns))
raw.head(3)


## 2. 删除泄漏列 `alive`


In [ ]:
leak_cols = [c for c in ['alive'] if c in raw.columns]
df = raw.drop(columns=leak_cols)
if leak_cols:
    print('dropped leak columns:', leak_cols)
if 'alive' in raw.columns:
    print('corr(alive, survived) =', raw['alive'].map({'yes': 1, 'no': 0}).corr(raw['survived']))
print('remaining cols:', list(df.columns))


## 3. EDA：目标分布与缺失


In [ ]:
from wanglaoshi.analyze import DataAnalyzer

analyzer = DataAnalyzer(df)
display(analyzer.missing_value_analysis())
display(analyzer.basic_statistics().head(12))
print('overall survival rate:', df['survived'].mean().round(3))


### 3.1 生存率：Sex / Pclass


In [ ]:
rate_sex = df.groupby('sex')['survived'].agg(['mean', 'count'])
rate_class = df.groupby('class')['survived'].agg(['mean', 'count'])
print('by sex:\n', rate_sex)
print('by class:\n', rate_class)


### 3.2 生存率：Age 分箱 / Fare / Embarked


In [ ]:
import matplotlib.pyplot as plt

work = df.copy()
work['age_bin'] = pd.cut(work['age'], bins=[0, 12, 18, 35, 60, 100], labels=['child','teen','adult','mid','senior'])
print('by age_bin:\n', work.groupby('age_bin', observed=True)['survived'].mean())
print('by embarked:\n', work.groupby('embarked')['survived'].mean())
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
work.boxplot(column='fare', by='survived', ax=ax[0])
ax[0].set_title('Fare by survived')
work.groupby('embarked')['survived'].mean().plot(kind='bar', ax=ax[1], rot=0)
ax[1].set_title('Survival rate by embarked')
plt.suptitle('')
plt.tight_layout()
plt.show()


## 4. 划分训练/验证（stratify survived）

此后所有统计量（中位数、众数、频率编码）**只能来自训练集**。


In [ ]:
from sklearn.model_selection import train_test_split

y = df['survived']
idx_train, idx_valid = train_test_split(
    df.index, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
train_df = df.loc[idx_train].copy()
valid_df = df.loc[idx_valid].copy()
print('train survival', train_df['survived'].mean(), 'valid', valid_df['survived'].mean())
print('train size', len(train_df), 'valid size', len(valid_df))


## 5. 特征工程（fit on train only）


In [ ]:
import re

def add_engineered_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    # Title
    if 'name' in out.columns:
        out['title'] = out['name'].astype(str).apply(
            lambda s: re.search(r',\s*([^\.]+)\.', s).group(1).strip() if re.search(r',\s*([^\.]+)\.', s) else 'Unknown'
        )
    else:
        out['title'] = out['who'].astype(str)  # seaborn 版无 name，用 who 近似
    out['family_size'] = out['sibsp'].fillna(0) + out['parch'].fillna(0) + 1
    out['is_alone'] = (out['family_size'] == 1).astype(int)
    if 'deck' in out.columns:
        out['deck_clean'] = out['deck'].astype(str).fillna('Unknown')
    else:
        out['deck_clean'] = 'Unknown'
    return out

train_fe = add_engineered_features(train_df)
valid_fe = add_engineered_features(valid_df)
print('title counts (train top):\n', train_fe['title'].value_counts().head(8))


### 5.1 缺失填充：仅用训练集 Age median、Embarked mode


In [ ]:
age_med = train_fe['age'].median()
emb_mode = train_fe['embarked'].mode(dropna=True).iloc[0] if train_fe['embarked'].notna().any() else 'S'

for part in (train_fe, valid_fe):
    part['age'] = part['age'].fillna(age_med)
    part['embarked'] = part['embarked'].fillna(emb_mode)
    part['deck_clean'] = part['deck_clean'].replace('nan', 'Unknown')

print('NA after fill — train', train_fe['age'].isna().sum(), 'valid', valid_fe['age'].isna().sum())
print('age median (train-only):', age_med, 'embarked mode:', emb_mode)


### 5.2 类别频率编码：映射来自训练集


In [ ]:
cat_cols = ['sex', 'embarked', 'class', 'who', 'embark_town', 'title', 'deck_clean']
cat_cols = [c for c in cat_cols if c in train_fe.columns]

def freq_encode(tr: pd.DataFrame, va: pd.DataFrame, cols: list) -> tuple:
    tr = tr.copy()
    va = va.copy()
    for c in cols:
        freq = tr[c].astype(str).value_counts(normalize=True)
        tr[c + '_freq'] = tr[c].astype(str).map(freq).fillna(0)
        va[c + '_freq'] = va[c].astype(str).map(freq).fillna(0)
    return tr, va

train_enc, valid_enc = freq_encode(train_fe, valid_fe, cat_cols)
num_cols = ['age', 'sibsp', 'parch', 'fare', 'family_size', 'is_alone']
num_cols += [c for c in train_enc.columns if c.endswith('_freq')]
Xtr = train_enc[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
Xva = valid_enc[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
ytr, yva = train_fe['survived'], valid_fe['survived']
print('feature matrix', Xtr.shape)
Xtr.head(3)


## 6. 错误 vs 正确填充（叙事对比）


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ❌ 错误：在全表 df 上算 median 再填充后划分（信息泄漏 + 填充泄漏）
wrong_med = df['age'].median()
X_wrong = df.copy()
X_wrong['age'] = X_wrong['age'].fillna(wrong_med)
num_simple = ['age', 'sibsp', 'parch', 'fare']
Xw = X_wrong[num_simple]
yw = df['survived']
Xw_tr, Xw_va, yw_tr, yw_va = train_test_split(
    Xw, yw, test_size=0.25, random_state=RANDOM_STATE, stratify=yw
)
clf_wrong = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)
clf_wrong.fit(Xw_tr, yw_tr)
acc_wrong = accuracy_score(yw_va, clf_wrong.predict(Xw_va))

# ✅ 正确：本 Notebook 第 5 节的 Xtr/Xva
clf_right = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)
clf_right.fit(Xtr, ytr)
acc_right = accuracy_score(yva, clf_right.predict(Xva))
print(f'wrong-fill baseline accuracy: {acc_wrong:.3f}')
print(f'right-fill baseline accuracy: {acc_right:.3f}')
print('全表填充可能略乐观 —— 竞赛要用 train-only 统计')


## 7. LeakageDetector：可疑高相关特征


In [ ]:
from wanglaoshi import compete as CT

detector = CT.LeakageDetector()
leaks = detector.detect_target_leakage(Xtr, ytr, threshold=0.85, method='correlation')
print('suspicious features (corr>0.85):', leaks if leaks else 'none — good')
# 若仍有 alive 残留列会在此告警


## 8. QuickBaseline


In [ ]:
baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr, ytr, Xva, yva, task_type='classification')
comparison = baseline.compare_models(results, top_n=6)
comparison


### 7.1 FeatureEngineering 模块（可选辅助）


In [ ]:
from wanglaoshi import features as FE

eng = FE.AdvancedFeatureEngineer()
# 演示：对训练集数值列做安全预览（不在此替换主流程）
preview = train_enc[num_cols].copy()
print('AdvancedFeatureEngineer loaded:', type(eng).__name__)
print('numeric feature preview shape', preview.shape)


### 7.2 生存率可视化汇总


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 7))
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[0, 0], rot=0, color=['#4477AA', '#CC6677'])
axes[0, 0].set_title('Survival by sex')
df.groupby('class')['survived'].mean().plot(kind='bar', ax=axes[0, 1], rot=45)
axes[0, 1].set_title('Survival by class')
df.groupby('embarked')['survived'].mean().plot(kind='bar', ax=axes[1, 0], rot=0)
axes[1, 0].set_title('Survival by embarked')
df.assign(age_filled=df['age'].fillna(df['age'].median())).boxplot(
    column='fare', by='survived', ax=axes[1, 1]
)
axes[1, 1].set_title('Fare by survived')
plt.suptitle('Titanic survival EDA panel')
plt.tight_layout()
plt.show()


### 7.3 缺失 Age 与生存（训练集视角）


In [ ]:
train_plot = train_df.copy()
train_plot['age_missing'] = train_plot['age'].isna()
print(train_plot.groupby('age_missing')['survived'].agg(['mean', 'count']))
print('→ Age 缺失可能与生存相关 —— 应用训练集中位数填充而非 0')


## 9. 模拟提交（Survived 列）


In [ ]:
from pathlib import Path

best = comparison.iloc[0]['Model']
pred = results[best]['predictions']
out = Path('outputs/02_titanic')
out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'PassengerId': np.arange(1, len(pred) + 1), 'Survived': yva.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
sub.head()


### 9.1 StratifiedKFold 线下分（均值 ± 标准差）


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

full_X = pd.concat([Xtr, Xva], axis=0)
full_y = pd.concat([ytr, yva], axis=0)
rf = RandomForestClassifier(n_estimators=80, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=3 if QUICK else 5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(rf, full_X, full_y, cv=cv, scoring='accuracy')
print(f'Titanic RF stratified CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')


### 9.2 特征分布对比（train vs valid）


In [ ]:
for col in ['fare', 'age', 'family_size']:
    print(col, 'train mean', round(train_fe[col].mean(), 2), 'valid mean', round(valid_fe[col].mean(), 2))
print('若 valid 与 train 分布差很大 —— 慎用过复杂的 train-only 映射')


## 10. 检查清单
- [x] Sex/Pclass/Age/Fare/Embarked 生存 EDA
- [x] 删除 alive 泄漏列
- [x] Title/FamilySize/IsAlone/Deck 工程（train-only 统计）
- [x] 错误 vs 正确 Age 填充对比
- [x] LeakageDetector
- [x] QuickBaseline + Survived 提交
- [ ] Cabin 文本深挖与折内目标编码


## 附录 A：缺失机制（MCAR / MAR / MNAR）速记

| 机制 | 含义 | Titanic 例子 |
| --- | --- | --- |
| MCAR | 缺失与一切无关 | 偶发记录遗漏 |
| MAR | 缺失依赖已观测变量 | Age 缺失率随舱等变化 |
| MNAR | 缺失依赖未观测/自身 | 「不愿填」与生存相关 |

正统做法：先按 Sex/Pclass **刻画缺失率**，再决定分组填充；**统计量只在训练集算**。参考 Kaggle 高赞 Titanic FE 教程中的 Title / FamilySize / Deck 工程。


In [ ]:
# 缺失率随舱等/性别
if 'age' in df.columns and 'class' in df.columns:
    print(df.assign(age_na=df['age'].isna()).groupby(['class','sex'])['age_na'].mean().unstack())


## 附录 B：启发式基线（gender model）

Kaggle `gender_submission` 的思想：女性几乎都标存活。任何 ML 模型应 **显著超过** 该启发式，否则特征工程无效。


In [ ]:
# 启发式：女性=1，男性=0（列名兼容 seaborn titanic）
sex_col = 'sex' if 'sex' in Xva.columns else None
if sex_col and 'sex' in work.columns:
    # 在验证折上用原始 sex
    sex_va = work.loc[Xva.index, 'sex'] if hasattr(Xva, 'index') else None
try:
    sex_va = work.loc[yva.index, 'sex']
    heur = (sex_va.astype(str).str.lower().isin(['female','woman'])).astype(int)
    from sklearn.metrics import accuracy_score
    print('gender heuristic acc on valid:', accuracy_score(yva, heur))
    print('model should beat this; else revisit features.')
except Exception as e:
    print('heuristic skipped:', e)
